# Notebook 3: Pandas — Data Wrangling Essentials
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Understand Series and DataFrame structures
- Load, inspect, and clean tabular data
- Filter, group, and aggregate data
- Merge and reshape DataFrames
- Apply lambda functions and custom transformations

## 1. Series and DataFrame

- A **Series** is a 1-D labelled array (like a column in a spreadsheet)
- A **DataFrame** is a 2-D table of Series (the spreadsheet itself)

Each axis has an **index** — labels that let you access data by name rather than integer position.

In [ ]:
import pandas as pd
import numpy as np

# Series
temps = pd.Series([22.5, 19.0, 24.3, 21.8, 17.2],
                  index=['Mon','Tue','Wed','Thu','Fri'],
                  name='Temperature (°C)')
print(temps)
print(f'\nMean temp: {temps.mean():.1f}°C')
print(f'Hottest day: {temps.idxmax()}')

# DataFrame from dict
data = {
    'Name'  : ['Alice','Bob','Carol','Dave','Eve'],
    'Age'   : [22, 25, 23, 28, 21],
    'Score' : [87.5, 91.0, 78.3, 95.1, 82.0],
    'Grade' : ['B','A','C','A','B'],
}
df = pd.DataFrame(data)
print('\n', df)

## 2. Inspecting Data

Always start an analysis with a quick inspection.

In [ ]:
# Use the built-in Titanic dataset via seaborn or a small synthetic one
import io
csv_data = '''PassengerId,Survived,Pclass,Name,Sex,Age,Fare
1,0,3,Braund,male,22,7.25
2,1,1,Cumings,female,38,71.28
3,1,3,Heikkinen,female,26,7.92
4,1,1,Futrelle,female,35,53.10
5,0,3,Allen,male,35,8.05
6,0,3,Moran,male,,8.46
7,0,1,McCarthy,male,54,51.86
8,0,3,Palsson,male,2,21.08
9,1,3,Johnson,female,27,11.13
10,1,2,Nasser,female,14,30.07
'''
df_t = pd.read_csv(io.StringIO(csv_data))

print('Shape       :', df_t.shape)
print('\nColumn types:\n', df_t.dtypes)
print('\nFirst 3 rows:\n', df_t.head(3))
print('\nBasic stats:\n', df_t.describe())

## 3. Selecting and Filtering Data

In [ ]:
# Select columns
print(df_t[['Name','Survived','Pclass']])

# Boolean filter
first_class = df_t[df_t['Pclass'] == 1]
print(f'\n1st class passengers: {len(first_class)}')

survivors_female = df_t[(df_t['Survived']==1) & (df_t['Sex']=='female')]
print(f'Female survivors: {len(survivors_female)}')

# .loc vs .iloc
print('\nRow by label (loc 2:4):\n', df_t.loc[2:4, ['Name','Age']])
print('\nRow by position (iloc 0:3, cols 0-2):\n', df_t.iloc[0:3, 0:3])

## 4. Handling Missing Data

In [ ]:
print('Missing values per column:\n', df_t.isnull().sum())
print(f'\n% missing Age: {df_t["Age"].isnull().mean()*100:.1f}%')

# Fill missing age with median
df_t['Age_filled'] = df_t['Age'].fillna(df_t['Age'].median())

# Drop rows with any missing value
df_dropped = df_t.dropna()
print(f'Rows before drop: {len(df_t)}, after: {len(df_dropped)}')

# Detect duplicates
print(f'Duplicate rows: {df_t.duplicated().sum()}')

## 5. GroupBy and Aggregation

In [ ]:
# Survival rate by class
print('Survival rate by Pclass:')
print(df_t.groupby('Pclass')['Survived'].mean().round(2))

# Multiple aggregations
print('\nAge stats by Sex:')
print(df_t.groupby('Sex')['Age'].agg(['mean','median','count']).round(1))

# Pivot table
pivot = df_t.pivot_table(values='Fare', index='Pclass', columns='Sex', aggfunc='mean')
print('\nAverage Fare (Pclass × Sex):\n', pivot.round(1))

## 6. Applying Functions and Creating New Columns

In [ ]:
# Vectorised operations
df_t['Fare_log'] = np.log1p(df_t['Fare'])

# Apply with lambda
df_t['Age_group'] = df_t['Age_filled'].apply(
    lambda a: 'Child' if a < 16 else ('Young' if a < 35 else 'Adult'))

print(df_t[['Name','Age_filled','Age_group','Fare_log']].head())

# Value counts
print('\nAge groups:\n', df_t['Age_group'].value_counts())

## 7. Merging and Reshaping

In [ ]:
# Merge (like SQL JOIN)
left = pd.DataFrame({'id':[1,2,3], 'name':['Alice','Bob','Carol']})
right = pd.DataFrame({'id':[1,2,4], 'score':[90,85,78]})

print('Inner join:')
print(pd.merge(left, right, on='id', how='inner'))
print('\nLeft join:')
print(pd.merge(left, right, on='id', how='left'))

# Melt (wide → long)
wide = pd.DataFrame({'student':['Alice','Bob'],
                     'math':[90,80], 'english':[85,88]})
long = pd.melt(wide, id_vars='student', var_name='subject', value_name='score')
print('\nMelted (wide → long):\n', long)

## Exercises

1. Using the titanic mini-dataset, compute the overall survival rate. Then break it down by `Sex` **and** `Pclass` together using `groupby`.
2. Add a column `'Fare_category'` that labels fares as `'Low'` (< 10), `'Medium'` (10–50), or `'High'` (> 50).
3. Sort the DataFrame by `Fare` descending and reset the index. Which passenger paid the most?
4. Load any CSV file of your choice with `pd.read_csv()`. Identify columns with > 30% missing values and drop them.

## Reflection Questions
- What is the difference between `.loc` and `.iloc`?
- When would you use `melt` vs `pivot_table`?
- Why is `groupby` often faster than writing an explicit loop?

---
**Next →** `04_data_visualization.ipynb`